# EXPT NO: 3 — Implementation of CNN for Image Classification
**Dataset:** CIFAR-10 (60,000 32x32 color images, 10 classes)

**Roll No:** _(edit this — add your roll number here)_

## PART A: Dataset Preparation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

# Load CIFAR-10 directly (no manual download needed)
(x_train_full, y_train_full), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

class_names = ['airplane','automobile','bird','cat','deer',
               'dog','frog','horse','ship','truck']

print("Full train shape:", x_train_full.shape)
print("Test shape:", x_test.shape)

In [ ]:
# Preprocess: normalize pixel values to [0,1]
x_train_full = x_train_full.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

y_train_full = y_train_full.flatten()
y_test = y_test.flatten()

# Split training set into train/validation (90/10)
val_split = int(0.9 * len(x_train_full))
x_train, x_val = x_train_full[:val_split], x_train_full[val_split:]
y_train, y_val = y_train_full[:val_split], y_train_full[val_split:]

print("Train:", x_train.shape, "Val:", x_val.shape, "Test:", x_test.shape)

In [ ]:
# Visualize a few sample images
plt.figure(figsize=(8,8))
for i in range(9):
    plt.subplot(3,3,i+1)
    plt.imshow(x_train[i])
    plt.title(class_names[y_train[i]])
    plt.axis('off')
plt.tight_layout()
plt.show()

## PART B: CNN Model Implementation

In [ ]:
model = models.Sequential([
    layers.Input(shape=(32,32,3)),

    layers.Conv2D(32, (3,3), activation='relu', padding='same'),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(64, (3,3), activation='relu', padding='same'),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(128, (3,3), activation='relu', padding='same'),
    layers.MaxPooling2D((2,2)),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(10, activation='softmax')
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

## PART C: Model Training and Evaluation

In [ ]:
history = model.fit(
    x_train, y_train,
    epochs=10,
    batch_size=64,
    validation_data=(x_val, y_val)
)

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=2)
print(f"Training accuracy (final epoch):   {history.history['accuracy'][-1]:.4f}")
print(f"Validation accuracy (final epoch): {history.history['val_accuracy'][-1]:.4f}")
print(f"Training loss (final epoch):       {history.history['loss'][-1]:.4f}")
print(f"Validation loss (final epoch):     {history.history['val_loss'][-1]:.4f}")
print(f"Testing accuracy:                  {test_acc:.4f}")
print(f"Testing loss:                      {test_loss:.4f}")

In [ ]:
# Accuracy vs Epoch
plt.figure(figsize=(6,4))
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Accuracy vs Epoch')
plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.legend(); plt.grid(True)
plt.show()

# Loss vs Epoch
plt.figure(figsize=(6,4))
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Loss vs Epoch')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend(); plt.grid(True)
plt.show()

## PART D: Performance Analysis

In [ ]:
y_pred_probs = model.predict(x_test)
y_pred = np.argmax(y_pred_probs, axis=1)

print(classification_report(y_test, y_pred, target_names=class_names))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.title('Confusion Matrix')
plt.show()

In [ ]:
# Sample correct predictions
plt.figure(figsize=(10,6))
correct_idx = np.where(y_pred == y_test)[0]
for i, idx in enumerate(correct_idx[:9]):
    plt.subplot(3,3,i+1)
    plt.imshow(x_test[idx])
    plt.title(f"Pred: {class_names[y_pred[idx]]}\nActual: {class_names[y_test[idx]]}", fontsize=8)
    plt.axis('off')
plt.suptitle("Sample Correct Predictions")
plt.tight_layout()
plt.show()

In [ ]:
# Misclassified images
plt.figure(figsize=(10,6))
wrong_idx = np.where(y_pred != y_test)[0]
for i, idx in enumerate(wrong_idx[:9]):
    plt.subplot(3,3,i+1)
    plt.imshow(x_test[idx])
    plt.title(f"Pred: {class_names[y_pred[idx]]}\nActual: {class_names[y_test[idx]]}", fontsize=8, color='red')
    plt.axis('off')
plt.suptitle("Misclassified Images")
plt.tight_layout()
plt.show()

print(f"Total misclassified: {len(wrong_idx)} out of {len(y_test)}")

## Analysis Summary

- **Classification accuracy:** see test accuracy printed above.
- **Confusion matrix:** shows which classes are most confused (e.g., cat/dog, automobile/truck are common CNN confusions on CIFAR-10).
- **Strengths of CNNs:** automatically learn spatial hierarchies of features (edges → textures → object parts), far fewer parameters than a fully-connected network on raw pixels, translation-invariant via weight sharing and pooling.
- **Limitations:** need large labeled datasets, computationally expensive to train, sensitive to hyperparameters (learning rate, filter sizes, depth), can overfit on small datasets, struggles with rotated/occluded objects without augmentation.